In [ ]:
import pandas as pd
import numpy as np

# STEP 1: LOAD THE RAW DATA
print("Loading raw dataset...")
df_raw = pd.read_csv(
    "UNFCCC_v30.csv",
    encoding="utf-8-sig"    # handles the BOM character at the start of the file
)

print(f"  Raw shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"  Columns  : {df_raw.columns.tolist()}\n")

Loading raw dataset...
  Raw shape: 674,415 rows x 14 columns
  Columns  : ['Country_code', 'Country', 'Format_name', 'Pollutant_name', 'Sector_code', 'Sector_name', 'Parent_sector_code', 'Unit', 'Year', 'emissions', 'Notation', 'PublicationDate', 'DataSource', 'Country_code_3']



In [ ]:
# STEP 2: INITIAL EXPLORATION — understand what we have before cleaning
print("=== DATASET OVERVIEW ===")
print(f"  Year range    : {df_raw['Year'].min()} - {df_raw['Year'].max()}")
print(f"  Countries     : {df_raw['Country'].nunique()} unique")
print(f"  Gas types     : {df_raw['Pollutant_name'].unique().tolist()}")
print(f"  Null emissions: {df_raw['emissions'].isna().sum():,} rows")
print()

=== DATASET OVERVIEW ===
  Year range    : 1990 - 2024
  Countries     : 32 unique
  Gas types     : ['CH4', 'N2O', 'All greenhouse gases - (CO2 equivalent)', 'CO2', 'SF6 - (CO2 equivalent)', 'NF3 - (CO2 equivalent)', 'HFCs - (CO2 equivalent)', 'Unspecified mix of HFCs and PFCs - (CO2 equivalent)', 'PFCs - (CO2 equivalent)']
  Null emissions: 267,293 rows



In [ ]:
# STEP 3: DROP COLUMNS WE DO NOT NEED
# Columns removed and why:
#   Format_name        -> always "IPCC Common Reporting Format" — constant, useless
#   DataSource         -> always "EEA" — constant, useless
#   PublicationDate    -> admin metadata, not needed for analysis
#   Country_code       -> 2-letter code; we keep Country_code_3 (ISO 3-letter for maps)
#   Notation           -> only explained nulls; no longer needed after we drop nulls
#   Parent_sector_code -> sector hierarchy already captured in Sector_code itself

columns_to_drop = [
    "Format_name", "DataSource", "PublicationDate",
    "Country_code", "Notation", "Parent_sector_code"
]

df = df_raw.drop(columns=columns_to_drop)
print(f"After dropping unused columns: {df.shape[1]} columns remaining\n")

After dropping unused columns: 8 columns remaining



In [ ]:
# STEP 4: RENAME COLUMNS — clean, Pythonic, dashboard-friendly names

df = df.rename(columns={
    "Country"        : "country",
    "Country_code_3" : "country_code",
    "Pollutant_name" : "gas",
    "Sector_code"    : "sector_code",
    "Sector_name"    : "sector",
    "Unit"           : "unit",
    "Year"           : "year",
    "emissions"      : "emissions_raw"
})

print("Columns renamed.\n")

Columns renamed.



In [ ]:
# STEP 5: REMOVE EU-27 AGGREGATE ROWS

# The dataset includes "EU-27" as a pre-aggregated entity.
# We keep only INDIVIDUAL countries to avoid double-counting in all charts.
before = len(df)
df = df[df["country"] != "EU-27"].copy()
print(f"Removed EU-27 rows: {before - len(df):,} dropped | {len(df):,} remaining\n")


Removed EU-27 rows: 22,470 dropped | 651,945 remaining



In [ ]:
# STEP 6: DROP NULL EMISSION ROWS (Notation rows)
# Drop all rows where emissions_raw is NaN.
# These are the Notation-coded rows (NO, IE, NA, NE) with no numerical value.

before = len(df)
df = df[df["emissions_raw"].notna()].copy()
print(f"Dropped null emission rows: {before - len(df):,} dropped | {len(df):,} remaining\n")

Dropped null emission rows: 263,898 dropped | 388,047 remaining



In [ ]:
# STEP 7: ENSURE EMISSIONS IS NUMERIC
# Coerce any stray non-numeric values to NaN then drop them (edge case safety)

df["emissions_raw"] = pd.to_numeric(df["emissions_raw"], errors="coerce")
before = len(df)
df = df[df["emissions_raw"].notna()].copy()
print(f"Dropped non-numeric rows: {before - len(df)} removed | {len(df):,} remaining\n")

Dropped non-numeric rows: 0 removed | 388,047 remaining



In [ ]:
# STEP 8: FILTER TO RELEVANT YEAR RANGE (2000–2024)

# We keep 2000–2024 (25 years) rather than the full 1990–2024 range.
# Reasons:
#   1. 1990s data for Eastern European countries is inconsistent — many were
#      transitioning economies with unreliable reporting standards
#   2. Our dashboard audience (decision-makers) cares about recent trends,
#      not 35-year history
#   3. 25 years still covers the full policy story:
#      pre-Paris (2000-2014) → Paris Agreement (2015) → COVID dip (2020)
#      → post-COVID recovery (2021-2024)
#   4. Cleaner, more readable charts with a focused time window

YEAR_START = 2000
YEAR_END   = 2024

before = len(df)
df = df[(df["year"] >= YEAR_START) & (df["year"] <= YEAR_END)].copy()
print(f"Filtered to {YEAR_START}-{YEAR_END}: {before - len(df):,} rows removed | {len(df):,} remaining\n")

Filtered to 2000-2024: 108,549 rows removed | 279,498 remaining



In [ ]:
# STEP 9: CLASSIFY SECTOR LEVELS
# The sector_code column is hierarchical. We add a new column 'sector_level'
# to flag how granular each row is:
#
#   "total"     -> aggregated country totals (sector_code starts with "Sectors/")
#   "top_level" -> 6 main sector categories identified by single-digit codes: 1-6
#   "detailed"  -> all sub-sector breakdowns (e.g. 1.A, 1.A.3, 1.A.3.b.i)

TOP_LEVEL_CODES = {"1", "2", "3", "4", "5", "6"}

def classify_sector_level(code):
    """Classify each row as total, top_level, or detailed based on sector_code."""
    code = str(code).strip()
    if code.startswith("Sectors/"):
        return "total"
    elif code in TOP_LEVEL_CODES:
        return "top_level"
    else:
        return "detailed"

df["sector_level"] = df["sector_code"].apply(classify_sector_level)

print("Sector level distribution:")
print(df["sector_level"].value_counts().to_string())
print()

Sector level distribution:
sector_level
detailed     250975
top_level     17555
total         10968



In [ ]:
# STEP 10: CLEAN SECTOR NAMES

# Strip any extra whitespace
df["sector"] = df["sector"].str.strip()

# Rename the 6 top-level sectors to clean short names for dashboard display
TOP_SECTOR_NAMES = {
    "1. Energy"                                  : "Energy",
    "2. Industrial processes and product use"    : "Industrial Processes",
    "3. Agriculture"                             : "Agriculture",
    "4. Land use, land-use change and forestry"  : "Land Use (LULUCF)",
    "5. Waste"                                   : "Waste",
    "6.  Other"                                  : "Other"
}
df["sector"] = df["sector"].replace(TOP_SECTOR_NAMES)

# Consolidate the four totals variants into two clean labels
# (with indirect / without indirect are both valid totals — we merge them)
TOTALS_SECTOR_NAMES = {
    "Sectors/Totals Total (without LULUCF)"                 : "Total (excl. LULUCF)",
    "Sectors/Totals Total (with LULUCF)"                    : "Total (incl. LULUCF)",
    "Sectors/Totals Total (without LULUCF, with indirect)"  : "Total (excl. LULUCF)",
    "Sectors/Totals Total (with LULUCF, with indirect)"     : "Total (incl. LULUCF)",
}
df["sector"] = df["sector"].replace(TOTALS_SECTOR_NAMES)

print("Sector names cleaned.")
print(f"  Top-level sectors: {df[df['sector_level']=='top_level']['sector'].unique().tolist()}\n")

Sector names cleaned.
  Top-level sectors: ['Land Use (LULUCF)', 'Energy', 'Agriculture', 'Waste', 'Industrial Processes', 'Other']



In [ ]:
# STEP 11: SIMPLIFY GAS NAMES
# Shorten long gas type labels for clean display in charts and dropdown filters

GAS_NAME_MAP = {
    "All greenhouse gases - (CO2 equivalent)"            : "All GHGs",
    "SF6 - (CO2 equivalent)"                             : "SF6",
    "NF3 - (CO2 equivalent)"                             : "NF3",
    "HFCs - (CO2 equivalent)"                            : "HFCs",
    "PFCs - (CO2 equivalent)"                            : "PFCs",
    "Unspecified mix of HFCs and PFCs - (CO2 equivalent)": "HFCs+PFCs mix"
}
df["gas"] = df["gas"].replace(GAS_NAME_MAP)

print(f"Gas names simplified: {df['gas'].unique().tolist()}\n")

Gas names simplified: ['CH4', 'N2O', 'All GHGs', 'CO2', 'SF6', 'NF3', 'HFCs+PFCs mix', 'HFCs', 'PFCs']



In [ ]:
# STEP 12: CONVERT EMISSIONS UNITS
# Original unit is Gigagrams (Gg).
#   1 Gg = 1,000 tonnes = 0.001 Mt (megatonnes)
#
# Rows in "Gg CO2 equivalent" are converted to Mt CO2 eq.
# (megatonnes is the standard unit used in climate policy and news media)
#
# Rows in plain "Gg" (individual gas mass e.g. CH4 in Gg) stay as Gg.
#
# NOTE: Negative values (e.g. LULUCF carbon sinks) are VALID — do not drop them.

df["emissions_mt"] = df.apply(
    lambda row: round(row["emissions_raw"] * 0.001, 6)
    if row["unit"] == "Gg CO2 equivalent"
    else round(row["emissions_raw"], 6),
    axis=1
)

df["unit"] = df["unit"].replace({
    "Gg CO2 equivalent" : "Mt CO2 eq.",
    "Gg"                : "Gg"
})

print("Emissions converted: Gg CO2 eq. -> Mt CO2 eq. where applicable.\n")


Emissions converted: Gg CO2 eq. -> Mt CO2 eq. where applicable.



In [ ]:
# STEP 13: FINAL COLUMN SELECTION & ORDER

df_final = df[[
    "country",        # Country name (e.g. Germany)
    "country_code",   # ISO 3-letter code (e.g. DEU) — used for choropleth maps
    "year",           # Year (1990-2024)
    "sector_level",   # total / top_level / detailed
    "sector_code",    # Original code (kept for reliable filtering)
    "sector",         # Clean sector name
    "gas",            # Gas type (CO2, CH4, N2O, All GHGs, etc.)
    "unit",           # Mt CO2 eq. or Gg
    "emissions_mt"    # Final emissions value
]].copy()

print(f"Final master dataset: {df_final.shape[0]:,} rows x {df_final.shape[1]} columns")
print(df_final.head(5).to_string())
print()

Final master dataset: 279,498 rows x 9 columns
    country country_code  year sector_level sector_code           sector  gas unit  emissions_mt
0   Finland          FIN  2014     detailed   1.A.3.b.i  1.A.3.b.i. Cars  CH4   Gg       0.52735
1  Slovenia          SVN  2003     detailed   1.A.3.b.i  1.A.3.b.i. Cars  CH4   Gg       0.53275
2  Bulgaria          BGR  2022     detailed   1.A.3.b.i  1.A.3.b.i. Cars  CH4   Gg       0.54293
3  Bulgaria          BGR  2020     detailed   1.A.3.b.i  1.A.3.b.i. Cars  CH4   Gg       0.54624
4   Belgium          BEL  2023     detailed   1.A.3.b.i  1.A.3.b.i. Cars  CH4   Gg       0.54691



In [ ]:
# STEP 14: BUILD 3 FOCUSED SUB-DATASETS FOR THE DASHBOARD
# Each dataset powers a specific part of the Streamlit dashboard.
# Keeping them separate makes the app faster and the code cleaner.

# ── 14a: df_totals — Country-level totals over time ──────────────────────────
# Powers: time-series line chart, country ranking bars, choropleth map
# Filter: country total rows (excl. LULUCF) + All GHGs combined measure
# Unit: Mt CO2 eq.

df_totals = df_final[
    (df_final["sector_code"] == "Sectors/Totals_excl") &
    (df_final["gas"] == "All GHGs")
].copy()

# Deduplicate: if two rows exist for same country/year, keep the row with
# the larger absolute emissions value, as this is likely the more complete measure.
df_totals = (
    df_totals
    .sort_values("emissions_mt", ascending=False)
    .drop_duplicates(subset=["country", "year"], keep="first")
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

print(f"df_totals  (country totals by year)  : {len(df_totals):,} rows")
print(f"  Countries : {df_totals['country'].nunique()}")
print(f"  Year range: {df_totals['year'].min()} - {df_totals['year'].max()}")
print(f"  Unit      : {df_totals['unit'].unique().tolist()}")
print(f"  Nulls     : {df_totals['emissions_mt'].isna().sum()}")
print(f"  Sample Germany:\n"
      f"{df_totals[df_totals['country']=='Germany'][['country','year','emissions_mt']].head(5).to_string()}\n")


# ── 14b: df_sectors — Sector breakdown per country per year ──────────────────
# Powers: stacked bar charts and sector comparison charts
# Filter: top-level sectors only + All GHGs
# Unit: Mt CO2 eq.

df_sectors = df_final[
    (df_final["sector_level"] == "top_level") &
    (df_final["gas"] == "All GHGs")
].copy()

df_sectors = (
    df_sectors
    .sort_values(["country", "year", "sector"])
    .reset_index(drop=True)
)

print(f"df_sectors (sector breakdown)        : {len(df_sectors):,} rows")
print(f"  Countries: {df_sectors['country'].nunique()}")
print(f"  Year range: {df_sectors['year'].min()} - {df_sectors['year'].max()}")
print(f"  Sectors  : {sorted(df_sectors['sector'].unique().tolist())}")
print(f"  Unit     : {df_sectors['unit'].unique().tolist()}")
print(f"  Nulls    : {df_sectors['emissions_mt'].isna().sum()}\n")


# ── 14c: df_gases — Gas type breakdown per country over time ─────────────────
# Powers: gas-specific trend charts
# Filter: country totals (excl. LULUCF) + individual gas types
# Exclude "All GHGs" aggregate and "HFCs+PFCs mix" to avoid double-counting.
#
# IMPORTANT:
# df_gases contains mixed units:
#   - CO2, CH4 and N2O are measured in Gg
#   - HFCs, PFCs, SF6 and NF3 are measured in Mt CO2 eq.
# Therefore, rename emissions_mt to emissions_value for this file only.

df_gases = df_final[
    (df_final["sector_code"] == "Sectors/Totals_excl") &
    (~df_final["gas"].isin(["All GHGs", "HFCs+PFCs mix"]))
].copy()

df_gases = (
    df_gases
    .rename(columns={"emissions_mt": "emissions_value"})
    .sort_values(["country", "year", "gas"])
    .reset_index(drop=True)
)

print(f"df_gases   (gas type breakdown)      : {len(df_gases):,} rows")
print(f"  Countries: {df_gases['country'].nunique()}")
print(f"  Year range: {df_gases['year'].min()} - {df_gases['year'].max()}")
print(f"  Gas types: {sorted(df_gases['gas'].unique().tolist())}")
print(f"  Units    : {df_gases['unit'].unique().tolist()}")
print(f"  Nulls    : {df_gases['emissions_value'].isna().sum()}\n")


# STEP 15: FINAL SANITY CHECKS

print("=== FINAL SANITY CHECKS ===")

# Germany 2000 total should be around 1,000 Mt CO2 eq.
de_2000 = df_totals[
    (df_totals["country"] == "Germany") &
    (df_totals["year"] == 2000)
]["emissions_mt"].values

print(f"Germany 2000 total: {de_2000} Mt CO2 eq.  (expected around 1000-1100)")

# No duplicate country-year combinations in totals
dupes_totals = df_totals.duplicated(subset=["country", "year"]).sum()
print(f"Duplicate country-year rows in df_totals: {dupes_totals}  (expected 0)")

# No duplicate country-year-sector combinations in sectors
dupes_sectors = df_sectors.duplicated(subset=["country", "year", "sector"]).sum()
print(f"Duplicate country-year-sector rows in df_sectors: {dupes_sectors}  (expected 0)")

# No duplicate country-year-gas combinations in gases
dupes_gases = df_gases.duplicated(subset=["country", "year", "gas"]).sum()
print(f"Duplicate country-year-gas rows in df_gases: {dupes_gases}  (expected 0)")

# All 31 individual countries present
print(f"Countries in df_totals: {df_totals['country'].nunique()}  (expected 31)")
print(f"Countries in df_sectors: {df_sectors['country'].nunique()}  (expected 31)")
print(f"Countries in df_gases: {df_gases['country'].nunique()}  (expected 31)")

print("\nCountry list:")
print(sorted(df_totals["country"].unique().tolist()))

print("\n=== ROW COUNT SUMMARY ===")
print(f"  df_totals   : {len(df_totals):,} rows")
print(f"  df_sectors  : {len(df_sectors):,} rows")
print(f"  df_gases    : {len(df_gases):,} rows")
print(f"  df_final    : {len(df_final):,} rows")


# STEP 16: SAVE ALL CLEANED DATASETS TO CSV

df_totals.to_csv("df_totals.csv", index=False)
df_sectors.to_csv("df_sectors.csv", index=False)
df_gases.to_csv("df_gases.csv", index=False)
df_final.to_csv("df_full_clean.csv", index=False)

print("\n✅ All cleaned files saved successfully:")
print("   df_totals.csv      -> country totals over time; emissions_mt is Mt CO2 eq.")
print("   df_sectors.csv     -> sector breakdown per country/year; emissions_mt is Mt CO2 eq.")
print("   df_gases.csv       -> gas-specific values; emissions_value must be read with unit column")
print("   df_full_clean.csv  -> full cleaned master dataset")

df_totals  (country totals by year)  : 775 rows
  Countries : 31
  Year range: 2000 - 2024
  Unit      : ['Mt CO2 eq.']
  Nulls     : 0
  Sample Germany:
     country  year  emissions_mt
250  Germany  2000   1042.998220
251  Germany  2001   1057.082348
252  Germany  2002   1036.657998
253  Germany  2003   1030.538757
254  Germany  2004   1010.676326

df_sectors (sector breakdown)        : 3,900 rows
  Countries: 31
  Year range: 2000 - 2024
  Sectors  : ['Agriculture', 'Energy', 'Industrial Processes', 'Land Use (LULUCF)', 'Other', 'Waste']
  Unit     : ['Mt CO2 eq.']
  Nulls    : 0

df_gases   (gas type breakdown)      : 4,635 rows
  Countries: 31
  Year range: 2000 - 2024
  Gas types: ['CH4', 'CO2', 'HFCs', 'N2O', 'NF3', 'PFCs', 'SF6']
  Units    : ['Gg', 'Mt CO2 eq.']
  Nulls    : 0

=== FINAL SANITY CHECKS ===
Germany 2000 total: [1042.99822] Mt CO2 eq.  (expected around 1000-1100)
Duplicate country-year rows in df_totals: 0  (expected 0)
Duplicate country-year-sector rows in df_se